# Working with statistical plots

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

Every statistical builder (`Histogram`, `BarPlot`, `BoxPlot`, `ViolinPlot`,
`ScatterPlot`, `SeaLevelPlot`, `SeatsVotesPlot`, `PaintballPlot`) shares one set of
controls for labels, legends, reference lines, axes, and output. The format notebooks
show each plot in its most basic form; this page is where the shared customization
lives.

The examples use a small fake ensemble so the controls stay in the foreground.

In [ ]:
import numpy as np

from gerrytools.plotting import Histogram

rng = np.random.default_rng(3)
fake_scores = rng.normal(0.58, 0.05, size=2_000)
fake_plan_score = 0.66

plot = Histogram()
plot.add_dataset(fake_scores, facecolor="default_grey", edgecolor="black")
plot.add_vertical_lines(fake_plan_score, linecolor="cherryblossompink", linewidth=2.0)
plot.show()

## Labels, titles, and figure size

`xlabel=`, `ylabel=`, `title=`, `figure_size=`, and `dpi=` are constructor arguments on
every builder. Pass `ax=` instead to render onto an existing Matplotlib axes (the
[Matplotlib guide](../composition.ipynb) covers multi-panel layouts).

In [ ]:
plot = Histogram(
    figure_size=(8, 4.5),
    xlabel="Fake ensemble score",
    ylabel="Plans",
    title="A fake ensemble",
)
plot.add_dataset(fake_scores, facecolor="default_grey", edgecolor="black")
plot.add_vertical_lines(fake_plan_score, linecolor="cherryblossompink", linewidth=2.0)
plot.show()

## Legends

Legends are disabled by default. Construct with `legend=True`, then give each dataset, line,
or band a meaningful `name=`. Some builders generate fallback labels for unnamed datasets,
but those labels describe insertion order rather than the data.

The default legend placement is outside the axes on the right. `set_legend_options()`
controls location, columns, frame, font size, and an optional title. In a multi-panel figure,
placing the legend inside an axes or reserving space in the Matplotlib layout prevents it from
overlapping the next panel. Use `save_legend(filepath)` to export a standalone key for a
report layout.

In [ ]:
plot = Histogram(xlabel="Fake ensemble score", legend=True)
plot.add_dataset(
    fake_scores,
    "2,000-plan fake ensemble",
    facecolor="default_grey",
    edgecolor="black",
)
plot.add_vertical_lines(
    fake_plan_score,
    name=f"Fake enacted plan: {fake_plan_score}",
    linecolor="cherryblossompink",
    linewidth=2.0,
)
plot.set_legend_options(loc="upper left", bbox_to_anchor=None, fontsize=8, fancybox=True)
plot.show()

It is also possible to save the legend separate from the plot:

```python
plot.save_legend("example_ensemble_legend.png")
```

![A legend for a histogram][example-legend]

[example-legend]: ../../../_static/images/example_ensemble_legend.png

## Reference lines, bands, and arrows

`add_vertical_lines()` and `add_horizontal_lines()` accept one value or several, plus
the usual `linecolor` / `linestyle` / `linewidth` options. `add_horizontal_band()` and
`add_vertical_band()` shade a range. `add_label_arrow()` and `add_text_arrow()` point
at a coordinate with a short label; the arrow direction is `"up"`, `"down"`,
`"left"`, or `"right"`.

In [ ]:
plot = Histogram(xlabel="Fake ensemble score")
plot.add_dataset(fake_scores, facecolor="default_grey", edgecolor="black")
plot.add_horizontal_band(10, 50, bandcolor="citizen_blue", linecolor=None, bandalpha=0.15)
plot.add_vertical_lines([0.5, 0.66], linecolor="black", linestyle="--")
plot.add_label_arrow((0.66, 90), "left", "fake plan")
plot.show()

`add_axis_text_arrow()` and `add_axis_label_arrow()` place those same arrows alongside
an axis to mark its direction of increase. Pick `"x"` or `"y"`, then nudge with
`position` (fraction along the axis) and `offset` (distance outside it); `direction`
defaults to `"right"` for x and `"up"` for y, and the opposite value reverses the
arrow. Set the common arrow face color, alpha, font color, and text rotation directly;
`text_options` and `arrow_options` collect the less common controls.

In [ ]:
plot = Histogram(xlabel="Fake ensemble score")
plot.add_dataset(fake_scores, facecolor="default_grey", edgecolor="black")
plot.add_axis_text_arrow("x", "more votes", fontcolor="white", position=0.13, offset=0.12)
plot.add_axis_label_arrow(
    "y",
    "more plans",
    position=0.9,
    offset=0.08,
    arrowfacecolor="citizen_blue",
    arrowfacealpha=0.8,
    fontcolor="black",
    textrotation=90,
)
plot.show()

## Limits and ticks

`set_xlim()` / `set_ylim()` fix the window. `set_xticks()` / `set_yticks()` take tick
locations and optional `labels=`; an empty list removes the axis entirely, the usual
report convention when only the distribution's shape matters.
`set_tick_style("x")` / `set_tick_style("y")` rotate and resize tick text, and
the underlying Matplotlib axes stays reachable through `.ax` for anything else, such
as percent formatting.

In [ ]:
import matplotlib.ticker as mticker
import numpy as np

plot = Histogram(xlabel="Fake ensemble score")
plot.add_dataset(fake_scores, facecolor="default_grey", edgecolor="black")
plot.set_bins(np.arange(0.4, 0.8, 0.01))
plot.set_xlim(0.4, 0.8)
plot.set_yticks([])
plot.set_tick_style("x", rotation=45, size=9)
plot.ax.xaxis.set_major_formatter(mticker.PercentFormatter(1))
plot.ax.figure

## Rebuilds and saving

Builders are lazy: accessing `.ax`, or calling `show()` or `save()`, builds the figure,
and later builder changes trigger a rebuild of the artists GerryTools owns. Ordinary
Matplotlib artists can share the axes; add them after the final rebuild if they must
stay on top. `save("figure.svg")` writes the figure; prefer SVG or PDF for line art.

## Related

- [All statistical plots](index.md)
- [Interacting with Matplotlib](../composition.ipynb) for shared axes and multi-panel layouts.
- [Working with geographic plots](../geographic/options.ipynb)
- [Plotting API](../../../api/plotting.rst)